In [2]:
import asyncio
import time
import json
import torch
from typing import List, Dict, Any, Optional, Union, Sequence
from dataclasses import dataclass
from pathlib import Path

# TensorRT-LLM imports
from tensorrt_llm.llmapi import LLM
from tensorrt_llm.sampling_params import SamplingParams
from tensorrt_llm.inputs.data import TextPrompt
from tensorrt_llm.executor import LoRARequest, PromptAdapterRequest
from tensorrt_llm.llmapi.llm_utils import KvCacheRetentionConfig
from tensorrt_llm.disaggregated_params import DisaggregatedParams
from tensorrt_llm.bindings.executor import KvCacheConfig

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/lib/python3.12/dist-packages/modelopt/torch/utils/import_utils.py:32: UserWarning: Failed to import huggingface plugin due to: AttributeError("module 'transformers.modeling_utils' has no attribute 'Conv1D'"). You may ignore this warning if you do not need this plugin.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/modelopt/torch/__init__.py:36: UserWarning: transformers version 4.56.0 is incompatible with nvidia-m

[TensorRT-LLM] TensorRT LLM version: 1.2.0rc1


In [3]:
@dataclass
class BatchRequest:
    """Structure for batch inference requests"""
    prompt: str
    request_id: str
    sampling_params: Optional[SamplingParams] = None
    metadata: Optional[Dict[str, Any]] = None
    lora_request: Optional[LoRARequest] = None
    prompt_adapter_request: Optional[PromptAdapterRequest] = None
    kv_cache_retention_config: Optional[KvCacheRetentionConfig] = None
    disaggregated_params: Optional[DisaggregatedParams] = None

In [4]:
class TRTLLMBatchInference:
    """
    TensorRT-LLM Batch Inference Handler
    Supports both synchronous and asynchronous batch processing
    """
    
    def __init__(
        self,
        model_path: Union[str, Path],
        tokenizer_path: Optional[Union[str, Path]] = None,
        tensor_parallel_size: int = 1,
        dtype: str = "auto",
        max_batch_size: Optional[int] = None,
        max_seq_len: Optional[int] = None,
        max_num_tokens: Optional[int] = None,
        trust_remote_code: bool = False,
        **kwargs
    ):
        """
        Initialize TensorRT-LLM batch inference engine
        
        Args:
            model_path: Path to the TensorRT-LLM engine or HuggingFace model
            tokenizer_path: Path to tokenizer (if different from model_path)
            tensor_parallel_size: Number of GPUs for tensor parallelism
            dtype: Model dtype (auto, half, float16, bfloat16, float32)
            max_batch_size: Maximum batch size
            max_seq_len: Maximum sequence length
            max_num_tokens: Maximum number of tokens
            trust_remote_code: Whether to trust remote code
            **kwargs: Additional arguments for LLM initialization
        """
        self.model_path = str(model_path)
        self.tensor_parallel_size = tensor_parallel_size

        #self.quant_config = QuantConfig()
        #self.quant_config.kv_cache_quant_algo = QuantAlgo.FP8

        #self.kv_cache_config = KvCacheConfig(free_gpu_memory_fraction=0.95

        self.kv_cache_config = {
            "free_gpu_memory_fraction": 0.95,
            "dtype": "fp8" # or the appropriate string value
        }
                                               
        # Initialize TensorRT-LLM engine
        self.llm = LLM(
            model=self.model_path,
            tokenizer=tokenizer_path,
            tensor_parallel_size=tensor_parallel_size,
            dtype=dtype,
            trust_remote_code=trust_remote_code,
            max_batch_size=max_batch_size,
            max_seq_len=8192,
            max_num_tokens=max_num_tokens,
            kv_cache_config=self.kv_cache_config,
            enable_attention_dp=True,
            **kwargs,
        )
        
    def _get_default_sampling_params(self) -> SamplingParams:
        """Get default sampling parameters with proper tokenizer setup"""
        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=4096
        )
        
        # Let the LLM's _prepare_sampling_params method handle tokenizer setup
        return self.llm._prepare_sampling_params(sampling_params)
    
    def batch_generate_sync(
        self, 
        prompts: List[str],
        sampling_params: Optional[SamplingParams] = None,
        batch_size: Optional[int] = None,
        use_tqdm: bool = True,
        lora_requests: Optional[List[LoRARequest]] = None,
        prompt_adapter_requests: Optional[List[PromptAdapterRequest]] = None,
        kv_cache_retention_configs: Optional[List[KvCacheRetentionConfig]] = None,
        disaggregated_params_list: Optional[List[DisaggregatedParams]] = None
    ) -> List[Dict[str, Any]]:
        """
        Synchronous batch generation
        
        Args:
            prompts: List of input prompts
            sampling_params: Sampling parameters for generation
            batch_size: If specified, process in chunks of this size
            use_tqdm: Whether to show progress bar
            lora_requests: List of LoRA requests (one per prompt)
            prompt_adapter_requests: List of prompt adapter requests
            kv_cache_retention_configs: List of KV cache retention configs
            disaggregated_params_list: List of disaggregated parameters
            
        Returns:
            List of generation results
        """
        if sampling_params is None:
            sampling_params = self._get_default_sampling_params()

        print(sampling_params)
        
        results = []
        
        if batch_size is None:
            # Process all prompts at once
            start_time = time.time()
            
            # Prepare inputs for TensorRT-LLM
            inputs = [TextPrompt(prompt=prompt) for prompt in prompts]
            
            outputs = self.llm.generate(
                inputs=inputs,
                sampling_params=sampling_params,
                use_tqdm=use_tqdm,
                lora_request=lora_requests,
                prompt_adapter_request=prompt_adapter_requests,
                kv_cache_retention_config=kv_cache_retention_configs,
                disaggregated_params=disaggregated_params_list
            )
            
            end_time = time.time()
            
            # Process outputs
            if not isinstance(outputs, list):
                outputs = [outputs]
                
            for i, output in enumerate(outputs):
                results.append(self._format_output(output, i, end_time - start_time))
        else:
            # Process in chunks
            for i in range(0, len(prompts), batch_size):
                batch_prompts = prompts[i:i + batch_size]
                batch_lora_requests = lora_requests[i:i + batch_size] if lora_requests else None
                batch_prompt_adapter_requests = prompt_adapter_requests[i:i + batch_size] if prompt_adapter_requests else None
                batch_kv_cache_configs = kv_cache_retention_configs[i:i + batch_size] if kv_cache_retention_configs else None
                batch_disaggregated_params = disaggregated_params_list[i:i + batch_size] if disaggregated_params_list else None
                
                start_time = time.time()
                
                inputs = [TextPrompt(prompt=prompt) for prompt in batch_prompts]
                
                outputs = self.llm.generate(
                    inputs=inputs,
                    sampling_params=sampling_params,
                    use_tqdm=use_tqdm,
                    lora_request=batch_lora_requests,
                    prompt_adapter_request=batch_prompt_adapter_requests,
                    kv_cache_retention_config=batch_kv_cache_configs,
                    disaggregated_params=batch_disaggregated_params
                )
                
                end_time = time.time()
                
                if not isinstance(outputs, list):
                    outputs = [outputs]
                
                for j, output in enumerate(outputs):
                    results.append(self._format_output(output, i + j, end_time - start_time))
        
        return results
    
    async def batch_generate_async(
        self,
        batch_requests: List[BatchRequest],
        max_concurrent: int = 100
    ) -> List[Dict[str, Any]]:
        """
        Asynchronous batch generation with request management
        
        Args:
            batch_requests: List of BatchRequest objects
            max_concurrent: Maximum number of concurrent requests
            
        Returns:
            List of generation results
        """
        # Semaphore to limit concurrent requests
        semaphore = asyncio.Semaphore(max_concurrent)
        
        async def process_single_request(request: BatchRequest) -> Dict[str, Any]:
            async with semaphore:
                sampling_params = request.sampling_params or self._get_default_sampling_params()
                
                start_time = time.time()
                
                # Create input for TensorRT-LLM
                inputs = TextPrompt(prompt=request.prompt)
                
                # TensorRT-LLM generate_async returns a RequestOutput future
                try:
                    output = self.llm.generate_async(
                        inputs=inputs,
                        sampling_params=sampling_params,
                        lora_request=request.lora_request,
                        prompt_adapter_request=request.prompt_adapter_request,
                        kv_cache_retention_config=request.kv_cache_retention_config,
                        disaggregated_params=request.disaggregated_params,
                        streaming=False
                    )
                    
                    # Wait for completion
                    result = output.result()
                    
                    end_time = time.time()
                    
                    return {
                        'prompt': result.prompt,
                        'generated_text': result.outputs[0].text,
                        'finish_reason': result.outputs[0].finish_reason.name if hasattr(result.outputs[0].finish_reason, 'name') else str(result.outputs[0].finish_reason),
                        'tokens_generated': len(result.outputs[0].token_ids) if hasattr(result.outputs[0], 'token_ids') else None,
                        'prompt_tokens': len(result.prompt_token_ids) if hasattr(result, 'prompt_token_ids') else None,
                        'request_id': request.request_id,
                        'generation_time': end_time - start_time,
                        'metadata': request.metadata
                    }
                    
                except Exception as e:
                    return {
                        'request_id': request.request_id,
                        'error': str(e),
                        'metadata': request.metadata
                    }
        
        # Process all requests concurrently
        tasks = [process_single_request(req) for req in batch_requests]
        results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Handle exceptions
        processed_results = []
        for i, result in enumerate(results):
            if isinstance(result, Exception):
                processed_results.append({
                    'request_id': batch_requests[i].request_id,
                    'error': str(result),
                    'metadata': batch_requests[i].metadata
                })
            else:
                processed_results.append(result)
        
        return processed_results
    
    async def batch_generate_streaming(
        self,
        batch_requests: List[BatchRequest],
        callback_fn: Optional[callable] = None
    ):
        """
        Streaming batch generation (yields results as they complete)
        
        Args:
            batch_requests: List of BatchRequest objects
            callback_fn: Optional callback function for each completed result
        """
        async def stream_generator():
            results = await self.batch_generate_async(batch_requests)
            for result in results:
                if callback_fn:
                    callback_fn(result)
                yield result
        
        return stream_generator()
    
    def _format_output(self, output, request_idx: int, generation_time: float) -> Dict[str, Any]:
        """Format TensorRT-LLM output to match VLLM format"""
        return {
            'prompt': output.prompt,
            'generated_text': output.outputs[0].text,
            'finish_reason': output.outputs[0].finish_reason.name if hasattr(output.outputs[0].finish_reason, 'name') else str(output.outputs[0].finish_reason),
            'tokens_generated': len(output.outputs[0].token_ids) if hasattr(output.outputs[0], 'token_ids') else None,
            'prompt_tokens': len(output.prompt_token_ids) if hasattr(output, 'prompt_token_ids') else None,
            'request_id': f"trtllm_{request_idx}",
            'generation_time': generation_time
        }
    
    def get_stats(self, timeout: Optional[float] = 2) -> List[dict]:
        """Get iteration statistics from the runtime"""
        return self.llm.get_stats(timeout=timeout)
    
    async def get_stats_async(self, timeout: Optional[float] = 2):
        """Get iteration statistics asynchronously"""
        return self.llm.get_stats_async(timeout=timeout)
    
    def save_engine(self, engine_dir: str) -> None:
        """Save the built engine to the given path"""
        self.llm.save(engine_dir)
    
    def shutdown(self) -> None:
        """Shutdown the inference engine"""
        if hasattr(self.llm, 'shutdown'):
            self.llm.shutdown()

In [5]:
def benchmark_batch_inference(
    model_path: str,
    test_prompts: List[str],
    batch_sizes: List[int] = [1, 4, 8, 16, 32],
    **llm_kwargs
) -> Dict[str, Any]:
    """
    Benchmark different batch sizes for inference performance
    
    Args:
        model_path: Path to the TensorRT-LLM engine or model
        test_prompts: List of test prompts
        batch_sizes: Different batch sizes to test
        **llm_kwargs: Additional arguments for TRTLLMBatchInference
        
    Returns:
        Benchmark results
    """
    inference_engine = TRTLLMBatchInference(model_path, **llm_kwargs)
    results = {}
    
    for batch_size in batch_sizes:
        print(f"Testing batch size: {batch_size}")
        start_time = time.time()
        
        # Repeat prompts to match batch size if needed
        test_batch = (test_prompts * ((batch_size // len(test_prompts)) + 1))[:batch_size]
        
        outputs = inference_engine.batch_generate_sync(
            test_batch,
            batch_size=batch_size
        )
        
        end_time = time.time()
        
        total_tokens = sum(r.get('tokens_generated', 0) for r in outputs if r.get('tokens_generated') is not None)
        total_time = end_time - start_time
        
        results[f'batch_{batch_size}'] = {
            'total_time': total_time,
            'requests_per_second': batch_size / total_time if total_time > 0 else 0,
            'tokens_per_second': total_tokens / total_time if total_time > 0 else 0,
            'avg_tokens_per_request': total_tokens / batch_size if batch_size > 0 else 0,
            'memory_usage': torch.cuda.memory_allocated() if torch.cuda.is_available() else 0
        }
    
    return results


In [6]:
if __name__ == "__main__":
    # Update model path for TensorRT-LLM
    model_path = "/workspace/models/Qwen3-30B-A3B-Instruct-2507-FP8/"  # This can be engine path or HF model path
    
    input_file = "/workspace/jupyter-notebooks/lab-2-offline-inference/single_prompt_6647_tokens.json"
    
    # Read input JSON file
    with open(input_file, "r") as f:
        data = json.load(f)

    test_prompt = data["input_text"]
    '''
    # Run synchronous demo
    print("=== SYNCHRONOUS BATCH DEMO ===")
    demo_sync_batch(model_path)
    '''
    '''
    # Run asynchronous demo
    print("\n=== ASYNCHRONOUS BATCH DEMO ===")
    asyncio.run(demo_async_batch(model_path))
    '''
    # Run benchmark
    print("\n=== PERFORMANCE BENCHMARK ===")
    test_prompts = [test_prompt]
    
    benchmark_results = benchmark_batch_inference(
        model_path,
        test_prompts,
        batch_sizes=[1, 2, 4, 8, 32, 64, 128, 256]
    )
    
    for batch_config, metrics in benchmark_results.items():
        print(f"{batch_config}: {metrics['requests_per_second']:.2f} req/s, "
              f"{metrics['tokens_per_second']:.2f} tokens/s")


=== PERFORMANCE BENCHMARK ===
[10/29/2025-09:34:35] [TRT-LLM] [I] Using LLM with PyTorch backend
[10/29/2025-09:34:35] [TRT-LLM] [W] Using default gpus_per_node: 1
[10/29/2025-09:34:35] [TRT-LLM] [I] Set PluginConfig.nccl_plugin to None.
[10/29/2025-09:34:35] [TRT-LLM] [I] neither checkpoint_format nor checkpoint_loader were provided, checkpoint_format will be set to HF.


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
You are using a model of type qwen3_moe to instantiate a model of type . This is not supported for all configurations of models and can yield errors.
rank 0 using MpiPoolSession to spawn MPI processes
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Multiple distributions found for package optimum. Picked distribution: optimum
/usr/local/lib/python3.12/dist-packages/modelopt/torch/utils/import_utils.py:32: UserWarning: Failed to import huggingface plugin due to: AttributeError("module 'transformers.modeling_utils' has no attribute 'Conv1D'"). You may ignore this warning if you do not need this plugin.
  warnings.warn(
/u

[TensorRT-LLM] TensorRT LLM version: 1.2.0rc1
[TensorRT-LLM][INFO] Refreshed the MPI local session


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "schema" in "ResponseFormat" shadows an attribute in parent "OpenAIBaseModel"
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1113/1113 [00:06<00:00, 163.45it/s]


Model init total -- 14.02s
[TensorRT-LLM][INFO] Max KV cache blocks per sequence: 257 [window size=8193], tokens per block=32, primary blocks=258, secondary blocks=0, max sequence length=8193
[TensorRT-LLM][INFO] Number of tokens per block: 32.
[TensorRT-LLM][INFO] [MemUsageChange] Allocated 0.38 GiB for max tokens in paged KV cache (8256).
[TensorRT-LLM][WARNING] Attention workspace size is not enough, increase the size from 0 bytes to 67200512 bytes
[TensorRT-LLM][WARNING] Attention workspace size is not enough, increase the size from 67200512 bytes to 86507520 bytes
[TensorRT-LLM][WARNING] [kv cache manager] storeContextBlocks: Can not find sequence for request 2048
[TensorRT-LLM][WARNING] [kv cache manager] storeContextBlocks: Can not find sequence for request 2049
[TensorRT-LLM][INFO] Max KV cache blocks per sequence: 257 [window size=8193], tokens per block=32, primary blocks=30178, secondary blocks=0, max sequence length=8193
[TensorRT-LLM][INFO] Number of tokens per block: 32.


...

Testing batch size: 128
SamplingParams(end_id=151645, pad_id=151643, max_tokens=4096, bad=None, bad_token_ids=None, stop=None, stop_token_ids=None, include_stop_str_in_output=False, embedding_bias=None, logits_processor=None, apply_batched_logits_processor=False, n=1, best_of=1, use_beam_search=False, top_k=None, top_p=0.9, top_p_min=None, top_p_reset_ids=None, top_p_decay=None, seed=None, temperature=0.7, min_tokens=None, beam_search_diversity_rate=None, repetition_penalty=None, presence_penalty=None, frequency_penalty=None, length_penalty=None, early_stopping=None, no_repeat_ngram_size=None, min_p=None, beam_width_array=None, logprobs=None, prompt_logprobs=None, return_context_logits=False, return_generation_logits=False, exclude_input_from_output=True, return_encoder_output=False, return_perf_metrics=False, additional_model_outputs=None, _context_logits_auto_enabled=False, _generation_logits_auto_enabled=False, _return_log_probs=False, lookahead_config=None, guided_decoding=None, ig

Processed requests: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:39<00:00,  3.26it/s]


Testing batch size: 256
SamplingParams(end_id=151645, pad_id=151643, max_tokens=4096, bad=None, bad_token_ids=None, stop=None, stop_token_ids=None, include_stop_str_in_output=False, embedding_bias=None, logits_processor=None, apply_batched_logits_processor=False, n=1, best_of=1, use_beam_search=False, top_k=None, top_p=0.9, top_p_min=None, top_p_reset_ids=None, top_p_decay=None, seed=None, temperature=0.7, min_tokens=None, beam_search_diversity_rate=None, repetition_penalty=None, presence_penalty=None, frequency_penalty=None, length_penalty=None, early_stopping=None, no_repeat_ngram_size=None, min_p=None, beam_width_array=None, logprobs=None, prompt_logprobs=None, return_context_logits=False, return_generation_logits=False, exclude_input_from_output=True, return_encoder_output=False, return_perf_metrics=False, additional_model_outputs=None, _context_logits_auto_enabled=False, _generation_logits_auto_enabled=False, _return_log_probs=False, lookahead_config=None, guided_decoding=None, ig

Processed requests: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:00<00:00,  4.21it/s]


batch_1: 0.10 req/s, 154.50 tokens/s
batch_2: 0.19 req/s, 291.86 tokens/s
batch_4: 0.35 req/s, 536.16 tokens/s
batch_8: 0.59 req/s, 912.77 tokens/s
batch_32: 1.53 req/s, 2364.40 tokens/s
batch_64: 2.33 req/s, 3593.33 tokens/s
batch_128: 3.03 req/s, 4682.83 tokens/s
batch_256: 3.88 req/s, 5990.25 tokens/s
